# Imports

In [ ]:
import pandas as pd
import os
import glob
# import mahotas
from pathlib import Path
import numpy as np
from tqdm import tqdm
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

#from transformers import AutoImageProcessor, TFViTModel

from tensorflow.keras.applications import ResNet50, VGG19, ConvNeXtBase, DenseNet121, DenseNet169
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.applications.convnext import preprocess_input
from tensorflow.keras.applications.densenet import preprocess_input

import cv2


from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.utils import load_img, img_to_array

from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from typing import List, Union, Literal
import skimage

from scipy.signal import convolve2d
from scipy.spatial.distance import cdist

In [ ]:
benignant_images_path = "./data/benignant_images"
malignant_images_path = "./data/malignant_images"

In [ ]:
benignant_images = []
malignant_images = []
for file in glob.glob(os.path.join(benignant_images_path, "*.jpg")):
    benignant_images.append(file)

for file in glob.glob(os.path.join(malignant_images_path, "*.jpg")):
    malignant_images.append(file)

# Functions

In [ ]:
def glcm(img : np.ndarray, 
         distances : Union[List[int],np.ndarray] =[1,3,5], 
         angles : Union[List[float],np.ndarray] = np.deg2rad([0,90,180,270])):
    
    assert isinstance(img, np.ndarray) and len(img.shape) == 2
    hists = graycomatrix(img, distances=distances, angles=angles, normed=True, symmetric=True)
    prop_names = ["contrast", "dissimilarity", "homogeneity", "ASM", "energy", "correlation"]
    props = np.array([ graycoprops(hists, prop).flatten() for prop in prop_names]).flatten()
    
    return props

In [ ]:
def feature_extract_glcm(images_path, save_filename):
    rows = []
    glcm_features = 0
    for images in images_path:
        # Read image in grayscale
        img = cv2.imread(images, cv2.IMREAD_GRAYSCALE)
    
        # Compute LBP
        glcm_features = glcm(img)
    
        # Create row: filename + features
        row = [Path(images).name] + glcm_features.tolist()
        rows.append(row)
    
    # Create column names: first column is "filename", rest are "f0, f1, ..."
    columns = ["filename"] + [f"f{i}" for i in range(len(glcm_features))]
    
    # Build DataFrame
    df = pd.DataFrame(rows, columns=columns)
    os.makedirs(os.path.join(os.getcwd(),"features"), exist_ok=True)
    df.to_csv(os.path.join(os.path.join(os.getcwd(), "features"),"glcm_"+save_filename+".csv"), index=False)
    return df

In [ ]:
benignant_glcm = feature_extract_glcm(benignant_images, "benignant_features")
malignant_glcm = feature_extract_glcm(malignant_images, "malignant_features")

In [ ]:
def lbp(image : np.ndarray, 
        P : int = 8, 
        R : int = 2, 
        method : Literal['default', 'ror', 'uniform', 'nri_uniform', 'var'] = 'nri_uniform'):
        
    assert isinstance(image, np.ndarray) and len(image.shape) == 2
    desc = local_binary_pattern(image, P, R, method=method)
    n_bins = int(desc.max() + 1)
    hist, _ = np.histogram(desc, density=True, bins=n_bins, range=(0, n_bins))

    return hist

In [ ]:
def feature_extract_lbp(images_path, save_filename):
    rows = []
    glcm_features = 0
    for images in images_path:
        # Read image in grayscale
        img = cv2.imread(images, cv2.IMREAD_GRAYSCALE)
    
        # Compute LBP
        lbp_features = lbp(img)
    
        # Create row: filename + features
        row = [Path(images).name] + lbp_features.tolist()
        rows.append(row)
    
    # Create column names: first column is "filename", rest are "f0, f1, ..."
    columns = ["filename"] + [f"f{i}" for i in range(len(lbp_features))]
    
    # Build DataFrame
    df = pd.DataFrame(rows, columns=columns)
    os.makedirs(os.path.join(os.getcwd(),"features"), exist_ok=True)
    df.to_csv(os.path.join(os.path.join(os.getcwd(), "features"),"lbp_"+save_filename+".csv"), index=False)
    return df

In [ ]:
benignant_lbp = feature_extract_lbp(benignant_images, "benignant_features")
malignant_lbp = feature_extract_lbp(malignant_images, "malignant_features")

In [ ]:
def lpq(img,winSize=7, decorr=1, mode='nh'):
    rho=0.90

    STFTalpha=1/winSize  # alpha in STFT approaches (for Gaussian derivative alpha=1)

    convmode='valid' # Compute descriptor responses only on part that have full neigborhood. Use 'same' if all pixels are included (extrapolates np.image with zeros).

    img=np.float64(img) # Convert np.image to double
    r=(winSize-1)/2 # Get radius from window size
    x=np.arange(-r,r+1)[np.newaxis] # Form spatial coordinates in window

    #  STFT uniform window
    #  Basic STFT filters
    w0=np.ones_like(x)
    w1=np.exp(-2*np.pi*x*STFTalpha*1j)
    w2=np.conj(w1)

    ## Run filters to compute the frequency response in the four points. Store np.real and np.imaginary parts separately
    # Run first filter
    filterResp1=convolve2d(convolve2d(img,w0.T,convmode),w1,convmode)
    filterResp2=convolve2d(convolve2d(img,w1.T,convmode),w0,convmode)
    filterResp3=convolve2d(convolve2d(img,w1.T,convmode),w1,convmode)
    filterResp4=convolve2d(convolve2d(img,w1.T,convmode),w2,convmode)

    # Initilize frequency domain matrix for four frequency coordinates (np.real and np.imaginary parts for each frequency).
    freqResp=np.dstack([filterResp1.real, filterResp1.imag,
                        filterResp2.real, filterResp2.imag,
                        filterResp3.real, filterResp3.imag,
                        filterResp4.real, filterResp4.imag])

    if decorr == 1:
        xp, yp = np.meshgrid(np.arange(1, winSize + 1), np.arange(1, winSize + 1))
        pp = np.column_stack((yp.flatten(), xp.flatten()))
        dd = cdist(pp, pp)
        C = rho ** dd

        q1 = w0.reshape((winSize,1))@w1.reshape((1,winSize))
        q2 = w1.reshape((winSize,1))@w0.reshape((1,winSize))
        q3 = w1.reshape((winSize,1))@w1.reshape((1,winSize))
        q4 = w1.reshape((winSize,1))@w2.reshape((1,winSize))
        
        M = np.vstack((q1.real.T.ravel(), q1.imag.T.ravel(), q2.real.T.ravel(), q2.imag.T.ravel(),
                       q3.real.T.ravel(), q3.imag.T.ravel(), q4.real.T.ravel(), q4.imag.T.ravel()))
        
        D = np.dot(M,C).dot(M.T)
        A = np.diag([1.000007, 1.000006, 1.000005, 1.000004, 1.000003, 1.000002, 1.000001, 1])
        U, S, V = np.linalg.svd(np.dot(A, D).dot(A))
        V = V.T

        freqRespShape = freqResp.shape
        freqResp = freqResp.reshape((-1, freqResp.shape[2]))
        freqResp = np.dot(V.T, freqResp.T).T
        freqResp = freqResp.reshape(freqRespShape)
        freqRespDecorr = freqResp.copy()    
    
    ## Perform quantization and compute LPQ codewords
    inds = np.arange(freqResp.shape[2])[np.newaxis,np.newaxis,:]
    LPQdesc=((freqResp>0)*(2**inds)).sum(2)

    ## Switch format to uint8 if LPQ code np.image is required as output
    if mode=='im':
        LPQdesc=np.uint8(LPQdesc)

    ## Histogram if needed
    if mode=='nh' or mode=='h':
        LPQdesc=np.histogram(LPQdesc.flatten(),range(257))[0]

    ## Normalize histogram if needed
    if mode=='nh':
        LPQdesc=LPQdesc/LPQdesc.sum()

    return LPQdesc

In [ ]:
def feature_extract_lpq(images_path, save_filename):
    rows = []
    glcm_features = 0
    for images in images_path:
        # Read image in grayscale
        img = cv2.imread(images, cv2.IMREAD_GRAYSCALE)
    
        # Compute LBP
        lpq_features = lpq(img)
    
        # Create row: filename + features
        row = [Path(images).name] + lpq_features.tolist()
        rows.append(row)
    
    # Create column names: first column is "filename", rest are "f0, f1, ..."
    columns = ["filename"] + [f"f{i}" for i in range(len(lpq_features))]
    
    # Build DataFrame
    df = pd.DataFrame(rows, columns=columns)
    os.makedirs(os.path.join(os.getcwd(),"features"), exist_ok=True)
    df.to_csv(os.path.join(os.path.join(os.getcwd(), "features"),"lpq_"+save_filename+".csv"), index=False)
    return df

In [ ]:
benignant_lpq = feature_extract_lpq(benignant_images, "benignant_features")
malignant_lpq = feature_extract_lpq(malignant_images, "malignant_features")

## RESNET50

In [ ]:
def feature_extract_resnet50(images_path, save_filename, out_dir=None, verbose=False):
    """
    images_path: iterable de caminhos (strings ou Path) para imagens.
    save_filename: nome base do CSV de saída (sem extensão).
    out_dir: pasta para salvar, padrão ./features
    """
    if out_dir is None:
        out_dir = os.path.join(os.getcwd(), "features")
    os.makedirs(out_dir, exist_ok=True)

    # Carrega ResNet50 sem top, com pooling average => saída 2048-dim
    model = ResNet50(weights="imagenet", include_top=False, pooling="avg")

    filenames = []
    features_list = []

    for i, img_path in enumerate(images_path):
        # tenta obter filename (suporta str, Path ou objetos com .name/.filename)
        if isinstance(img_path, (str, Path)):
            filename = Path(img_path).name
        else:
            # se for PIL.Image ou array, tenta pegar atributo .filename/.name, se não, usar índice
            filename = getattr(img_path, "filename", None) or getattr(img_path, "name", None) or f"image_{i}"
        try:
            # se img_path for caminho, carregar da FS; caso contrário assumimos que é PIL.Image ou array
            if isinstance(img_path, (str, Path)):
                pil = load_img(img_path, target_size=(224, 224))
            else:
                # caso o usuário passe uma PIL.Image já carregada ou um numpy array
                pil = img_path  # load_img/img_to_array aceita PIL.Image; se for ndarray, img_to_array funciona abaixo

            x = img_to_array(pil)
            x = np.expand_dims(x, axis=0)
            x = preprocess_input(x)

            feat = model.predict(x, verbose=0)
            feat = feat.reshape(-1)  # 1D vetor

            filenames.append(filename)
            features_list.append(feat)

            if verbose and (i + 1) % 50 == 0:
                print(f"Processed {i+1} images")

        except Exception as e:
            # loga e pula imagem com problema
            print(f"Warning: failed to process {img_path}: {e}")
            continue

    if len(features_list) == 0:
        raise ValueError("No features extracted. Check your images_path list and image readability.")

    # Empilha em matriz (n_samples x features_dim)
    features_array = np.vstack(features_list)
    features_dim = features_array.shape[1]

    # Cria DataFrame de features e insere a coluna filename como primeira coluna
    feature_cols = [f"f{i}" for i in range(features_dim)]
    df_features = pd.DataFrame(features_array, columns=feature_cols)
    df_features.insert(0, "filename", filenames)

    # Salva CSV
    out_path = os.path.join(out_dir, f"resnet50_{save_filename}.csv")
    df_features.to_csv(out_path, index=False)

    if verbose:
        print(f"Saved features to {out_path} (shape={df_features.shape})")

    return df_features

In [ ]:
benignant_resnet50 = feature_extract_resnet50(benignant_images, "benignant_features")
malignant_resnet50 = feature_extract_resnet50(malignant_images, "malignant_features")

In [ ]:
benignant_resnet50 = pd.read_csv("./features/resnet50_benignant_features.csv")
malignant_resnet50 = pd.read_csv("./features/resnet50_malignant_features.csv")

In [ ]:
def feature_extract_vgg19(images_path, save_filename, out_dir=None, verbose=False):
    """
    Extrai features da camada 'fc2' da VGG19 (4096 dimensões).
    
    images_path: lista de caminhos de imagens
    save_filename: nome base do arquivo CSV
    out_dir: pasta para salvar (default = ./features)
    """
    if out_dir is None:
        out_dir = os.path.join(os.getcwd(), "features")
    os.makedirs(out_dir, exist_ok=True)

    # Carrega VGG19 pré-treinada, pegando a saída da camada 'fc2'
    base_model = VGG19(weights="imagenet")
    model = Model(inputs=base_model.input, outputs=base_model.get_layer("fc2").output)

    filenames = []
    features_list = []

    for i, img_path in enumerate(tqdm(images_path, desc="Extracting VGG19 Features")):
        if isinstance(img_path, (str, Path)):
            filename = Path(img_path).name
        else:
            filename = getattr(img_path, "filename", None) or getattr(img_path, "name", None) or f"image_{i}"
        try:
            if isinstance(img_path, (str, Path)):
                pil = load_img(img_path, target_size=(224, 224))
            else:
                pil = img_path

            x = img_to_array(pil)
            x = np.expand_dims(x, axis=0)
            x = preprocess_input(x)

            feat = model.predict(x, verbose=0)
            feat = feat.reshape(-1)  # vetor 1D (4096 valores)

            filenames.append(filename)
            features_list.append(feat)

            if verbose and (i + 1) % 50 == 0:
                print(f"Processed {i+1} images")

        except Exception as e:
            print(f"Warning: failed to process {img_path}: {e}")
            continue

    if len(features_list) == 0:
        raise ValueError("No features extracted. Check your images_path list and image readability.")

    # Empilha todas as features em matriz (n_samples x 4096)
    features_array = np.vstack(features_list)
    features_dim = features_array.shape[1]

    # Cria DataFrame de features
    feature_cols = [f"f{i}" for i in range(features_dim)]
    df_features = pd.DataFrame(features_array, columns=feature_cols)
    df_features.insert(0, "filename", filenames)

    # Salva CSV
    out_path = os.path.join(out_dir, f"vgg19_{save_filename}.csv")
    df_features.to_csv(out_path, index=False)

    if verbose:
        print(f"Saved features to {out_path} (shape={df_features.shape})")

    return df_features

In [ ]:
benignant_vgg19 = feature_extract_vgg19(benignant_images, "benignant_features")
malignant_vgg19 = feature_extract_vgg19(malignant_images, "malignant_features")